# Al Nakheel Trading Co. — Python EDA & Visualization

Day 3 of the project: connect to the cleaned `AlNakheelDB` SQL Server database (built in Days 1–2), pull `FactSales` into pandas, run a quick exploratory analysis, and visualize the same business questions answered in SQL — monthly revenue trend, revenue by branch, channel mix, and top products.

**Before running:** update the `server` variable in the connection cell below with your own SQL Server instance name.

## 1. Setup — install and import required libraries

- `pandas` — data as tables (DataFrames)
- `sqlalchemy` + `pyodbc` — connect to SQL Server from Python
- `matplotlib` + `seaborn` — charts
- `arabic_reshaper` + `python-bidi` — fix Arabic text rendering in matplotlib (without this, Arabic labels render reversed/disconnected)

In [ ]:
import sys
!{sys.executable} -m pip install -q pandas sqlalchemy pyodbc matplotlib seaborn arabic_reshaper python-bidi

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import urllib
from sqlalchemy import create_engine
import arabic_reshaper
from bidi.algorithm import get_display

sns.set_style("whitegrid")

def fix_arabic(text):
    """Reshape + reorder Arabic text so it renders correctly in matplotlib."""
    return get_display(arabic_reshaper.reshape(text))

## 2. Connect to SQL Server

Uses Windows Authentication (`Trusted_Connection=yes`) — no username/password needed. **You must replace `server` in the next cell with your own instance name** (find it in SSMS's "Server name" box, e.g. `localhost` or `YOURPC\\SQLEXPRESS`). The placeholder `YOUR_SERVER_NAME` is intentional — a real machine/instance name is personal to the developer's computer and isn't committed to the repo.

In [ ]:
server = 'YOUR_SERVER_NAME'   # <-- REQUIRED: replace with your own SQL Server instance name (see README)
database = 'AlNakheelDB'

params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={server};"
    f"DATABASE={database};"
    f"Trusted_Connection=yes;"
    f"Encrypt=yes;"
    f"TrustServerCertificate=yes;"
)

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")

# Sanity check: confirm the connection works and FactSales has the expected row count
test_query = "SELECT COUNT(*) AS RowCnt FROM dbo.FactSales;"
print(pd.read_sql(test_query, engine))

## 3. Load the cleaned sales data

Joins `FactSales` to `DimBranches` and `DimProducts` so branch and product **names** are available directly, instead of just IDs — makes grouping and plotting much more readable.

In [ ]:
query = """
SELECT
    fs.OrderID, fs.OrderDate, fs.CustomerName, fs.Quantity,
    fs.UnitPrice, fs.TotalAmount, fs.Channel, fs.PaymentMethod, fs.Status,
    db.BranchName,
    dp.ProductName, dp.Category
FROM dbo.FactSales fs
JOIN dbo.DimBranches db ON fs.BranchID = db.BranchID
JOIN dbo.DimProducts dp ON fs.ProductID = dp.ProductID;
"""

df = pd.read_sql(query, engine)
df.head()

## 4. Exploratory Data Analysis (EDA)

Quick shape/type/missing-value check. `CustomerName` is expected to have 15 missing values — same count found during SQL cleaning in Day 1 (left as NULL by design, not an error).

In [ ]:
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nMissing values per column:")
print(df.isnull().sum())
print("\nNumeric summary:")
df.describe()

`OrderDate` comes back from SQL Server as a plain string (`object` dtype). Convert it to a real datetime so it can be grouped/sorted chronologically.

In [ ]:
df['OrderDate'] = pd.to_datetime(df['OrderDate'])
df['YearMonth'] = df['OrderDate'].dt.to_period('M').astype(str)
print(df['OrderDate'].dtype)
print(df['OrderDate'].min(), "->", df['OrderDate'].max())

## 5. Branch name simplification (for chart labels)

`DimBranches.BranchName` contains full Arabic branch names (e.g. "فرع القاهرة - مدينة نصر"). For readable English chart labels, map each branch to a short English name.

**Important:** matched using `in` (substring search on a keyword like "قاهرة") rather than an exact-equality dictionary lookup. An exact-match dictionary silently produced `NaN` for one branch here — its raw name had an extra space that wasn't visible on screen, the same class of bug found in the Day 1 SQL cleaning. Substring matching avoids that failure mode entirely.

In [ ]:
def simplify_branch(name):
    if 'إسكندرية' in name:
        return 'Alex - Smouha'
    elif 'جيزة' in name:
        return 'Giza - Dokki'
    elif 'أسيوط' in name:
        return 'Assiut - Mahatta St.'
    elif 'منصورة' in name:
        return 'Mansoura - Gomhoreya St.'
    elif 'قاهرة' in name:
        return 'Cairo - Nasr City'
    return name

df['BranchLabel'] = df['BranchName'].apply(simplify_branch)

## 6. Chart 1 — Monthly Revenue Trend

In [ ]:
monthly_revenue = df.groupby('YearMonth')['TotalAmount'].sum().reset_index()

plt.figure(figsize=(14, 5))
plt.plot(monthly_revenue['YearMonth'], monthly_revenue['TotalAmount'], marker='o')
plt.xticks(rotation=45)
plt.title('Monthly Revenue')
plt.xlabel('Month')
plt.ylabel('Revenue (EGP)')
plt.tight_layout()
plt.savefig('charts/monthly_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Chart 2 — Total Revenue by Branch

In [ ]:
branch_revenue = df.groupby('BranchLabel')['TotalAmount'].sum().sort_values(ascending=False).reset_index()

plt.figure(figsize=(10, 5))
sns.barplot(data=branch_revenue, x='BranchLabel', y='TotalAmount', hue='BranchLabel', palette='Blues_d', legend=False)
plt.xticks(rotation=20)
plt.title('Total Revenue by Branch')
plt.xlabel('Branch')
plt.ylabel('Revenue (EGP)')
plt.tight_layout()
plt.savefig('charts/branch_revenue.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. Chart 3 — Revenue Share by Channel (Online vs In-Store)

In [ ]:
channel_revenue = df.groupby('Channel')['TotalAmount'].sum().reset_index()
channel_revenue['ChannelLabel'] = channel_revenue['Channel'].apply(
    lambda x: 'In-Store' if 'متجر' in x else 'Online'
)

plt.figure(figsize=(6, 6))
plt.pie(channel_revenue['TotalAmount'], labels=channel_revenue['ChannelLabel'],
        autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'], startangle=90)
plt.title('Revenue Share by Channel')
plt.tight_layout()
plt.savefig('charts/channel_share.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Chart 4 — Top 5 Products by Quantity Sold

Product names are kept in Arabic (unlike branches, these aren't easily translated 1:1), so `fix_arabic()` from the setup cell is used to render them correctly.

In [ ]:
top_products = df.groupby('ProductName')['Quantity'].sum().sort_values(ascending=False).head(5).reset_index()
top_products['ProductName_Fixed'] = top_products['ProductName'].apply(fix_arabic)

plt.figure(figsize=(10, 5))
sns.barplot(data=top_products, y='ProductName_Fixed', x='Quantity', hue='ProductName_Fixed', palette='Greens_d', legend=False)
plt.title('Top 5 Products by Quantity Sold')
plt.xlabel('Quantity Sold')
plt.ylabel('Product')
plt.tight_layout()
plt.savefig('charts/top_products.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

- In-store sales account for 77.6% of revenue vs 22.4% online.
- "جوال شاومي 14" is the top seller by unit volume, though not the top earner by revenue (see `sql/03_analysis/03_top_products_and_profitability.sql`).
- Branch revenue and monthly trends here match the SQL results from Day 2 — a useful cross-check that the SQL cleaning/aggregation logic is correct.

All charts are saved to `charts/` for use in the project README.